In [1]:
import numpy as np
import pandas as pd
import os
import sys
import time

In [2]:
import sklearn.tree
import sklearn.linear_model
import sklearn.metrics
import sklearn.ensemble
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import balanced_accuracy_score
from scipy.sparse import vstack, csr_matrix # Use vstack for sparse matrices

In [3]:

from pretty_print_sklearn_tree import pretty_print_sklearn_tree

In [4]:
# Plotting utils
import matplotlib
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8') # pretty matplotlib plots

import seaborn as sns
sns.set('notebook', font_scale=1.25, style='whitegrid')

In [5]:
def get_metrics(model, X, y):
    """Calculates balanced accuracy score for a given model and dataset."""
    yhat = model.predict(X)
    return balanced_accuracy_score(y, yhat)

# Load all data from train/valid/test

In [6]:
# --- Data Loading Helper Adapted for CSVs (Q3) ---

def load_bow_data(name):
    """Loads feature (X) and label (y) data from CSV files."""
    try:
        # Load features: x_train.csv, x_valid.csv, x_test.csv
        # The first row is the header (vocabulary).
        x_df = pd.read_csv(f'x_{name}.csv')
        vocab = x_df.columns.tolist()
        
        # Data points are the values below the header. Convert to a sparse matrix for efficiency.
        X = x_df.values
        X_sparse = csr_matrix(X)
        
        # Load labels: y_train.csv, y_valid.csv, y_test.csv
        # Labels should be a single column array
        y_df = pd.read_csv(f'y_{name}.csv')
        y = y_df.iloc[:, 0].values # Take the first (and only) column of data
        
        return X_sparse, y, vocab
    
    except FileNotFoundError as e:
        print(f"Error: {e}. Ensure {name} CSV files are in the current directory.")
        return None, None, None
    except Exception as e:
        print(f"An unexpected error occurred during data loading: {e}")
        return None, None, None

### Load training

In [7]:
x_tr_NF, y_tr_N, vocab_list_tr = load_bow_data('train')

### Load validation set

In [8]:
x_val_NF, y_val_N, vocab_list_val = load_bow_data('valid')

### Load test set 

In [9]:
x_test_NF, y_test_N, vocab_list_test = load_bow_data('test')

### Load vocabulary as a list of strings

In [10]:
# Use the vocabulary loaded from the training set features
vocab_list = vocab_list_tr
F_total = len(vocab_list)
print(f"Total features (vocabulary size): {F_total}")

Total features (vocabulary size): 7729


In [11]:
# Basic check
if x_tr_NF is not None:
    print(f"Train data shape: {x_tr_NF.shape}, Labels: {y_tr_N.shape}")

Train data shape: (6346, 7729), Labels: (6346,)


### Pack training and validation sets into big arrays (so we can use sklearn's hyperparameter search tools)

In [12]:
# Combine sparse matrices (vstack handles csr_matrix)
x_tr_val_NF = vstack([x_tr_NF, x_val_NF]) 
y_tr_val_N = np.hstack([y_tr_N, y_val_N])

In [13]:
# Create PredefinedSplit to use validation set as a fixed test fold
# 0 for train, -1 for validation (test fold)
split_test_fold = np.concatenate([
    np.zeros(x_tr_NF.shape[0]),
    -1 * np.ones(x_val_NF.shape[0])
])
my_splitter = PredefinedSplit(test_fold=split_test_fold)

In [14]:
print(f"Combined data shape: {x_tr_val_NF.shape}")

Combined data shape: (7138, 7729)


# Problem 1: Decision Trees

## 1A: Train a simple tree with depth 3

In [15]:
# Simple Decision Tree Classifier setup (Q3.2.1)
simple_tree = sklearn.tree.DecisionTreeClassifier(
    criterion='gini', 
    max_depth=3, 
    min_samples_leaf=1, 
    min_samples_split=2, 
    random_state=101
)

### **Fit the tree** 

**TODO Train on the training set** in the next coding cell

In [16]:
# Fit the simple tree on the training data
simple_tree.fit(x_tr_NF, y_tr_N)
print("Simple Decision Tree (max_depth=3) fitted.")

Simple Decision Tree (max_depth=3) fitted.


### **Figure 1: Print Tree** 

Use a helper function from the starter code

In [17]:
# Print the simple tree structure
print("Figure 1: Simple Decision Tree Structure (max_depth=3)")
pretty_print_sklearn_tree(simple_tree, feature_names=vocab_list)

Figure 1: Simple Decision Tree Structure (max_depth=3)
The binary tree structure has 15 nodes.
- depth   0 has    1 nodes, of which    0 are leaves
- depth   1 has    2 nodes, of which    0 are leaves
- depth   2 has    4 nodes, of which    0 are leaves
- depth   3 has    8 nodes, of which    8 are leaves
The decision tree:  (Note: Y = 'yes' to above question; N = 'no')
Decision: X['great'] <= 0.50?
  Y Decision: X['excel'] <= 0.50?
    Y Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.430 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.114 (1 total training examples)
    N Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.903 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.429 (1 total training examples)
  N Decision: X['return'] <= 0.50?
    Y Decision: X['bad'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.745 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.415 (1 total training examples

## 1B : Find best Decision Tree with grid search

In [18]:
# Define the parameter grid (Q3.2.2)
param_grid_dt = {
    'max_depth': [2, 8, 32, 128],
    'min_samples_leaf': [1, 3, 9],
    'random_state': [101] # Fixed for reproducibility
}

base_tree = sklearn.tree.DecisionTreeClassifier(
    criterion='gini', min_samples_split=2
)

In [19]:
# Perform Grid Search using the combined train/validation set and the PredefinedSplit
dt_searcher = GridSearchCV(
    estimator=base_tree,
    param_grid=param_grid_dt,
    scoring='balanced_accuracy',
    cv=my_splitter,  # Uses PredefinedSplit to define train/valid splits
    return_train_score=True,
    refit=False,
    verbose=2,
    n_jobs=-1
)

dt_searcher.fit(x_tr_val_NF, y_tr_val_N)
# Report best parameters
print("Best Decision Tree Hyperparameters found:")
print(dt_searcher.best_params_)

best_dt_params = dt_searcher.best_params_
print(f"\nBest max_depth: {best_dt_params['max_depth']}")
print(f"Best min_samples_leaf: {best_dt_params['min_samples_leaf']}")

Fitting 1 folds for each of 12 candidates, totalling 12 fits
Best Decision Tree Hyperparameters found:
{'max_depth': 32, 'min_samples_leaf': 1, 'random_state': 101}

Best max_depth: 32
Best min_samples_leaf: 1


### Build the best decision tree

**TODO Build the Best Tree on the training set** in the next coding cell



In [20]:
# Build the Best Tree (Best DT) using hyperparameters found by grid search
best_tree = sklearn.tree.DecisionTreeClassifier(
    criterion='gini',
    min_samples_split=2,
    **best_dt_params
)
best_tree.fit(x_tr_NF, y_tr_N)
print("Best Decision Tree built and fitted on training data.")

Best Decision Tree built and fitted on training data.


### Interpret the best decision tree

In [22]:
print("Printing structure for the Best Decision Tree:")
pretty_print_sklearn_tree(best_tree, feature_names=vocab_list)


Printing structure for the Best Decision Tree:
The binary tree structure has 925 nodes.
- depth   0 has    1 nodes, of which    0 are leaves
- depth   1 has    2 nodes, of which    0 are leaves
- depth   2 has    4 nodes, of which    0 are leaves
- depth   3 has    8 nodes, of which    0 are leaves
- depth   4 has   16 nodes, of which    5 are leaves
- depth   5 has   22 nodes, of which    6 are leaves
- depth   6 has   32 nodes, of which   14 are leaves
- depth   7 has   36 nodes, of which   21 are leaves
- depth   8 has   30 nodes, of which   14 are leaves
- depth   9 has   32 nodes, of which    7 are leaves
- depth  10 has   50 nodes, of which   28 are leaves
- depth  11 has   44 nodes, of which   23 are leaves
- depth  12 has   42 nodes, of which   21 are leaves
- depth  13 has   42 nodes, of which   21 are leaves
- depth  14 has   42 nodes, of which   23 are leaves
- depth  15 has   38 nodes, of which   21 are leaves
- depth  16 has   34 nodes, of which   15 are leaves
- depth  17

# Problem 2: Random forest

## 2A: Train a random forest with default settings

In [23]:
# Simple Random Forest setup (Q3.3.1)
simple_forest = sklearn.ensemble.RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_features='sqrt', # default for classifier is sqrt(F)
    max_depth=3, # Max depth 3 as specified
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=101,
    n_jobs=-1 # Use all available cores
)

### Fit the forest

**TODO Train on the training set** in the next coding cell

In [24]:
# Fit the simple forest on the training data
simple_forest.fit(x_tr_NF, y_tr_N)
print("Simple Random Forest (max_depth=3, n_estimators=100) fitted.")

Simple Random Forest (max_depth=3, n_estimators=100) fitted.


## 2B & Table 2: Feature Importances

In [30]:
# Access feature importances
importances = simple_forest.feature_importances_
indices = np.argsort(importances)[::-1]

# Top 10 words with highest feature importance
top_10_indices = indices[:10]
top_10_words = [(vocab_list[i], importances[i]) for i in top_10_indices]

# Terms with near-zero importance (importance <= 0.00001)
importance_threshold = 0.00001
unimportant_mask = importances <= importance_threshold
unimportant_indices = np.where(unimportant_mask)[0]
eligible_unimportant_count = len(unimportant_indices)

# Randomly chosen 10 terms with near-zero importance
prng = np.random.RandomState(42) # Fixed seed for reproducibility
if eligible_unimportant_count >= 10:
    rand_indices = prng.choice(unimportant_indices, size=10, replace=False)
else:
    rand_indices = unimportant_indices

rand_unimportant_words = [(vocab_list[i], importances[i]) for i in rand_indices]

# Prepare Table 2 for printing
important_words = [w[0] for w in top_10_words]
unimportant_words_list = [w[0] for w in rand_unimportant_words]
unimportant_words_list.extend(['-'] * (10 - len(unimportant_words_list))) # Pad to 10

print(f"Number of terms eligible for 'near-zero' importance (<= {importance_threshold}): {eligible_unimportant_count}")

Number of terms eligible for 'near-zero' importance (<= 1e-05): 7296


In [31]:
# Print Table 2 (Q3.3.1)
print("\nTable 2: Top 10 Important Words vs. Random 10 Unimportant Words")
print("|**Important Words**|**Unimportant Words**|")
print("|:-:|:-:|")
# The lists are guaranteed to be of length 10 or padded to 10.
for I, U in zip(important_words, unimportant_words_list):
    print(f"| {I} | {U} |")


Table 2: Top 10 Important Words vs. Random 10 Unimportant Words
|**Important Words**|**Unimportant Words**|
|:-:|:-:|
| return | had_not |
| excel | stylish |
| great | doubl |
| worst | link |
| poor | a_fairly |
| disappoint | story_of |
| your_money | the_temperature |
| i_love | we_had |
| the_best | again_and |
| don't | of_people |


## 2C: Best Random Forest via grid search



This block might take 2-10 minutes. 

If yours runs significantly longer, try this out on Google Colab instead.

In [48]:
# ===== FIXED LOADER (handles headers / index columns / strict numeric) =====
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import PredefinedSplit
from scipy import sparse as sp
# ---- Q3 patch: missing helpers ----
import numpy as np
from sklearn.metrics import balanced_accuracy_score

def _evaluate_bal_acc(model, Xtr, ytr, Xva, yva, Xte=None, yte=None):
    tr = balanced_accuracy_score(ytr, model.predict(Xtr))
    va = balanced_accuracy_score(yva, model.predict(Xva))
    te = None if Xte is None else balanced_accuracy_score(yte, model.predict(Xte))
    return tr, va, te

# Fallback feature names (if Cell 0 wasn't run)
if '_feature_names' not in globals() or _feature_names is None \
   or (hasattr(_feature_names, '__len__') and len(_feature_names) != X_tr.shape[1]):
    _feature_names = [f"term_{i}" for i in range(X_tr.shape[1])]

# Safe stack for train+val (used by optional cells)
def _stackX(A, B):
    try:
        import scipy.sparse as sp
        if sp.issparse(A) and sp.issparse(B):
            return sp.vstack([A, B], format='csr')
    except Exception:
        pass
    return np.vstack([A, B])


BASE_DIR = "."  # change if your CSVs are in a subfolder

FILE_MAP = {
    "x_train": "x_train.csv",
    "y_train": "y_train.csv",
    "x_valid": "x_valid.csv",
    "y_valid": "y_valid.csv",
    "x_test":  "x_test.csv",
    "y_test":  "y_test.csv",
}

def _path(name): return os.path.join(BASE_DIR, FILE_MAP[name])

def _drop_index_like(df: pd.DataFrame) -> pd.DataFrame:
    """Drop obvious index/id/Unnamed columns if present."""
    to_drop = []
    if len(df.columns) == 0:
        return df
    # candidates: first column if looks like index, plus any 'Unnamed: *'
    first = df.columns[0]
    cand_names = {"unnamed: 0", "index", "id"}
    if str(first).strip().lower() in cand_names:
        to_drop.append(first)
    # any column literally starting with 'Unnamed'
    to_drop += [c for c in df.columns if str(c).lower().startswith("unnamed")]
    # if first col is non-numeric and uniquely identifies rows (index-ish), drop it
    if first not in to_drop:
        s0 = df.iloc[:, 0]
        if (not np.issubdtype(s0.dtype, np.number)) and (s0.nunique() == len(s0)):
            to_drop.append(first)
    return df.drop(columns=list(dict.fromkeys(to_drop)))  # de-dup while preserving order

def _read_matrix(fp):
    # 1) Read with header (most common case)
    df = pd.read_csv(fp)
    df = _drop_index_like(df)

    # Convert anything numeric-looking to numeric
    conv = {}
    bad = []
    for c in df.columns:
        if np.issubdtype(df[c].dtype, np.number):
            continue
        # try numeric conversion
        coerced = pd.to_numeric(df[c], errors='coerce')
        if coerced.isnull().any():
            bad.append(c)
        else:
            conv[c] = coerced

    # binary strings true/false or yes/no often appear; map them if present
    if bad:
        mapping = {"true": 1, "false": 0, "yes": 1, "no": 0}
        still_bad = []
        for c in bad:
            s = df[c].astype(str).str.lower().map(mapping)
            if s.isnull().any():
                still_bad.append(c)
            else:
                conv[c] = s.astype(float)
        bad = still_bad

    # If columns are still non-numeric, it’s likely these are *not* feature columns.
    if bad:
        raise ValueError(f"Found non-numeric feature columns in {os.path.basename(fp)}: {bad}. "
                         f"Drop them if they are identifiers, or encode if truly categorical.")

    # apply conversions
    for c, s in conv.items():
        df[c] = s

    if not all(np.issubdtype(df[c].dtype, np.number) for c in df.columns):
        raise ValueError(f"Non-numeric dtypes remain in {os.path.basename(fp)} after cleaning.")

    return df.values.astype(np.float32)

def _read_vector(fp):
    s = pd.read_csv(fp).squeeze("columns")
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]
    # factorize if labels are strings
    if not np.issubdtype(s.dtype, np.number):
        codes, uniques = pd.factorize(s.astype(str))
        print("[info] Encoded labels -> integers mapping:")
        for i, u in enumerate(uniques):
            print(f"    {u!r} -> {i}")
        y = codes.astype(np.int32)
    else:
        y = s.astype(np.int32).values
    return y.ravel()

# ---- Load all splits
missing = [f for f in FILE_MAP.values() if not os.path.exists(os.path.join(BASE_DIR, f))]
if missing:
    raise FileNotFoundError(f"Missing CSVs in {os.path.abspath(BASE_DIR)}:\n{missing}")

x_train = _read_matrix(_path("x_train"))
y_train = _read_vector(_path("y_train"))
x_valid = _read_matrix(_path("x_valid"))
y_valid = _read_vector(_path("y_valid"))
x_test  = _read_matrix(_path("x_test"))
y_test  = _read_vector(_path("y_test"))

# ---- Sanity checks
assert x_train.shape[0] == y_train.shape[0]
assert x_valid.shape[0] == y_valid.shape[0]
assert x_test.shape[0]  == y_test.shape[0]
assert x_train.shape[1] == x_valid.shape[1] == x_test.shape[1], "Feature dimension mismatch"

print("Shapes:",
      f"\n  x_train: {x_train.shape}, y_train: {y_train.shape}",
      f"\n  x_valid: {x_valid.shape}, y_valid: {y_valid.shape}",
      f"\n  x_test : {x_test.shape},  y_test : {y_test.shape}")

# Canonical names used later
X_tr, y_tr = x_train, y_train
X_va, y_va = x_valid, y_valid
X_te, y_te = x_test,  y_test

# Predefined split for grid search (train=-1, val=0)
def _stackX(A, B):
    try:
        import scipy.sparse as sp
        if sp.issparse(A) and sp.issparse(B):
            return sp.vstack([A, B], format='csr')
    except Exception:
        pass
    return np.vstack([A, B])

X_cv = _stackX(X_tr, X_va)
y_cv = np.hstack([y_tr, y_va])
fold_ids = np.concatenate([
    -1 * np.ones(len(y_tr), dtype=int),
     0 * np.ones(len(y_va), dtype=int),
])
my_splitter = PredefinedSplit(fold_ids)

print("Prepared X_tr, y_tr, X_va, y_va, X_te, y_te and my_splitter.")


Shapes: 
  x_train: (6346, 7729), y_train: (6346,) 
  x_valid: (792, 7729), y_valid: (792,) 
  x_test : (793, 7729),  y_test : (793,)
Prepared X_tr, y_tr, X_va, y_va, X_te, y_te and my_splitter.


In [49]:
# ===== 3.1 Simple Random Forest =====
rf_simple = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    max_features='sqrt',   # "other hyperparams from starter code" may vary; sqrt is common default
    class_weight='balanced',
    random_state=101,
    n_jobs=-1,
)
rf_simple.fit(X_tr, y_tr)

# Balanced accuracy on splits
rf_simple_tr, rf_simple_va, rf_simple_te = _evaluate_bal_acc(
    rf_simple, X_tr, y_tr, X_va, y_va, X_te, y_te)

print(f"[RF simple] Balanced Acc — train: {rf_simple_tr:.4f} | val: {rf_simple_va:.4f} | test: {rf_simple_te:.4f}")

# ---- Feature importance tables ----
imp = rf_simple.feature_importances_
F = imp.shape[0]
names = np.asarray(_feature_names)

# Top-10 most important features
top_idx = np.argsort(imp)[::-1][:10]
top_df = pd.DataFrame({
    "rank": np.arange(1, len(top_idx)+1),
    "feature": names[top_idx],
    "importance": imp[top_idx],
})

# Near-zero importance (< 1e-5): sample 10 randomly (if available)
near_zero_mask = imp < 1e-5
eligible_idx = np.where(near_zero_mask)[0]
num_eligible = eligible_idx.size
sample_k = min(10, num_eligible)
rng = np.random.RandomState(SEED)
sampled_idx = rng.choice(eligible_idx, size=sample_k, replace=False) if sample_k > 0 else np.array([], dtype=int)

nearzero_df = pd.DataFrame({
    "feature": names[sampled_idx],
    "importance": imp[sampled_idx],
})
nearzero_df["eligibles_with_<1e-5"] = num_eligible

# Display as a two-panel table by concatenating side-by-side
two_panel = pd.concat(
    [top_df.set_index("rank"), nearzero_df.reset_index(drop=True)],
    axis=1
)
two_panel.columns = ["Top-10 feature", "Top-10 importance",
                     "Near-zero feature (<1e-5)", "Near-zero importance", "Eligible count"]
two_panel


[RF simple] Balanced Acc — train: 0.8210 | val: 0.7973 | test: 0.7829


,Top-10 feature,Top-10 importance,Near-zero feature (<1e-5),Near-zero importance,Eligible count
1,term_113,0.032995,term_2940,0.0,7296.0
2,term_100,0.029479,term_2720,0.0,7296.0
3,term_1,0.028982,term_6348,0.0,7296.0
4,term_283,0.028412,term_5062,0.0,7296.0
5,term_130,0.026751,term_6491,0.0,7296.0
6,term_68,0.024955,term_3774,0.0,7296.0
7,term_525,0.018005,term_6732,0.0,7296.0
8,term_167,0.017953,term_106,0.0,7296.0
9,term_78,0.017661,term_5760,0.0,7296.0
10,term_4,0.017659,NaN,NaN,NaN


### Do the search!

In [51]:
# ===== 3.2 Grid Search for Random Forest =====
# Per assignment:
param_grid_rf = {
    "max_features": [3, 10, 33, 100, 333],
    "max_depth": [16, 32],
    "min_samples_leaf": [1],
    "n_estimators": [100],
    "random_state": [101],
    # We'll also keep these consistent with the simple model
    # class_weight & n_jobs are fixed outside param_grid
}

rf_base = RandomForestClassifier(
    class_weight='balanced',
    n_jobs=-1,
    # other params will be set by GridSearchCV
)

rf_grid = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid_rf,
    scoring="balanced_accuracy",
    cv=my_splitter,             # uses provided train/val split
    return_train_score=True,
    refit=False,
    n_jobs=-1,
    verbose=1,
)

rf_grid.fit(X_cv, y_cv)

grid_results_rf = (
    pd.DataFrame(rf_grid.cv_results_)
      .sort_values("mean_test_score", ascending=False)
      .reset_index(drop=True)
)

best_row = grid_results_rf.iloc[0]
best_params_rf = {k.replace("param_", ""): best_row[f"param_{k}"] 
                  for k in param_grid_rf.keys()}

print("Best RF params (by validation balanced accuracy):", best_params_rf)
print(f"Best mean_test_score (val balanced acc): {best_row['mean_test_score']:.4f}")

# Train best RF on TRAIN only (to compare with simple RF on the same footing)
rf_best_val = RandomForestClassifier(
    **best_params_rf,
    class_weight='balanced',
    n_jobs=-1,
)
rf_best_val.fit(X_tr, y_tr)
rf_bestval_tr, rf_bestval_va, rf_bestval_te = _evaluate_bal_acc(
    rf_best_val, X_tr, y_tr, X_va, y_va, X_te, y_te)

print(f"[RF best (val-tuned)] Balanced Acc — train: {rf_bestval_tr:.4f} | val: {rf_bestval_va:.4f} | test: {rf_bestval_te:.4f}")

grid_results_rf.head(10)


Fitting 1 folds for each of 10 candidates, totalling 10 fits
Best RF params (by validation balanced accuracy): {'max_features': np.int64(33), 'max_depth': np.int64(32), 'min_samples_leaf': np.int64(1), 'n_estimators': np.int64(100), 'random_state': np.int64(101)}
Best mean_test_score (val balanced acc): 0.8499
[RF best (val-tuned)] Balanced Acc — train: 0.9646 | val: 0.8499 | test: 0.8408


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_max_features,param_min_samples_leaf,param_n_estimators,param_random_state,params,split0_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,mean_train_score,std_train_score
0,10.764716,0.0,1.110749,0.0,32,33,1,100,101,"{'max_depth': 32, 'max_features': 33, 'min_sam...",0.849916,0.849916,0.0,1,0.964561,0.964561,0.0
1,5.415627,0.0,0.518154,0.0,32,10,1,100,101,"{'max_depth': 32, 'max_features': 10, 'min_sam...",0.837029,0.837029,0.0,2,0.968810,0.968810,0.0
2,16.020177,0.0,0.744853,0.0,16,33,1,100,101,"{'max_depth': 16, 'max_features': 33, 'min_sam...",0.836455,0.836455,0.0,3,0.923286,0.923286,0.0
3,30.790812,0.0,0.760993,0.0,32,100,1,100,101,"{'max_depth': 32, 'max_features': 100, 'min_sa...",0.834197,0.834197,0.0,4,0.968487,0.968487,0.0
4,4.869648,0.0,0.406474,0.0,16,10,1,100,101,"{'max_depth': 16, 'max_features': 10, 'min_sam...",0.829808,0.829808,0.0,5,0.924697,0.924697,0.0
5,22.250640,0.0,0.516672,0.0,16,100,1,100,101,"{'max_depth': 16, 'max_features': 100, 'min_sa...",0.826822,0.826822,0.0,6,0.917281,0.917281,0.0
6,47.392098,0.0,0.060832,0.0,32,333,1,100,101,"{'max_depth': 32, 'max_features': 333, 'min_sa...",0.816104,0.816104,0.0,7,0.953500,0.953500,0.0
7,7.686517,0.0,0.843359,0.0,32,3,1,100,101,"{'max_depth': 32, 'max_features': 3, 'min_samp...",0.792947,0.792947,0.0,8,0.969435,0.969435,0.0
8,42.355352,0.0,0.634865,0.0,16,333,1,100,101,"{'max_depth': 16, 'max_features': 333, 'min_sa...",0.791722,0.791722,0.0,9,0.871387,0.871387,0.0
9,4.694261,0.0,0.424887,0.0,16,3,1,100,101,"{'max_depth': 16, 'max_features': 3, 'min_samp...",0.747997,0.747997,0.0,10,0.911281,0.911281,0.0


### Display search results

### 3.2 — Random Forest Tuning: Short Answers

**Q1. What is the maximum possible value for `max_features` for this dataset? Why is it beneficial to tune this hyperparameter?**  
- The maximum possible value is the **total number of input features** \(often denoted \(p\)\). Concretely:  
  **`max_features_max = p = X_train.shape[1]`**.  
  *(In text/BoW setups, \(p\) equals the vocabulary size.)*
- **Why tune it?** It controls the **strength–diversity trade-off** of the forest:  
  - **Smaller `max_features`** → more randomness per split → **less-correlated trees** → greater variance reduction when averaging → often **better generalization** (at the cost of slightly weaker individual trees).  
  - **Larger `max_features`** → stronger individual trees but **more correlation** across trees → averaging reduces variance less.  
  - The sweet spot is **data-dependent**, so we pick it via validation.

---

**Q2. When fitting random forests, what trade-off does `n_estimators` control? Can setting it very large cause overfitting? Why/why not?**  
- **Primary trade-off:** **variance reduction vs. compute**. Increasing `n_estimators` (number of trees) generally **reduces variance** and stabilizes predictions, but **increases training time and memory**, with **diminishing returns** after a point.  
- **Overfitting?** In standard Random Forests, as the number of trees \(T\) grows, test error **converges to an asymptote**; adding more trees **does not typically cause overfitting**—it just approaches the ensemble’s limiting performance. Practically, you stop when validation/OOB metrics **plateau**, since extra trees mainly add cost without improving accuracy. *(Note: more trees won’t fix high **bias** from shallow/over-regularized trees.)*


### Build the best random forest using the best hyperparameters found in 2B 

This is necessary so you have the specific best performing forest in your workspace.

Train *only* on training set (do not merge train and valid)


In [53]:
# ===== 3.3 Comparison table =====

rows = []

# Try to reuse Part 2 simple DT; else train it quickly
dt_simple = None
for cand in ['clf_dt_simple', 'dt_simple', 'dt_clf_simple']:
    if cand in globals():
        dt_simple = globals()[cand]
        break

if dt_simple is None:
    dt_simple = DecisionTreeClassifier(
        criterion='gini', max_depth=3, min_samples_leaf=1,
        min_samples_split=2, random_state=101, class_weight='balanced'
    )
    dt_simple.fit(X_tr, y_tr)

dt_s_tr, dt_s_va, dt_s_te = _evaluate_bal_acc(dt_simple, X_tr, y_tr, X_va, y_va, X_te, y_te)
rows.append({
    "Model": "Simple DT",
    "#Trees": 1,
    "max_depth": dt_simple.get_params().get("max_depth", None),
    "BA_train": dt_s_tr, "BA_val": dt_s_va, "BA_test": dt_s_te
})

# Best DT from Part 2.2 (if available); else try to re-find quickly using same grid.
dt_best = None
best_params_dt = None
for cand in ['best_params_dt', 'dt_best_params', 'dt_grid_best_params']:
    if cand in globals():
        best_params_dt = globals()[cand]
        break

if best_params_dt is None:
    # Re-run the small DT grid from 2.2 if needed
    dt_grid = GridSearchCV(
        DecisionTreeClassifier(class_weight='balanced'),
        param_grid={
            "max_depth": [2, 8, 32, 128],
            "min_samples_leaf": [1, 3, 9],
            "random_state": [101],
        },
        scoring="balanced_accuracy",
        cv=my_splitter,
        return_train_score=True,
        refit=False,
        n_jobs=-1,
        verbose=0,
    )
    dt_grid.fit(X_cv, y_cv)
    best_idx = np.argmax(dt_grid.cv_results_["mean_test_score"])
    best_params_dt = {
        "max_depth": dt_grid.cv_results_[f"param_max_depth"][best_idx],
        "min_samples_leaf": dt_grid.cv_results_[f"param_min_samples_leaf"][best_idx],
        "random_state": dt_grid.cv_results_[f"param_random_state"][best_idx],
    }

dt_best = DecisionTreeClassifier(class_weight='balanced', **best_params_dt)
dt_best.fit(X_tr, y_tr)
dt_b_tr, dt_b_va, dt_b_te = _evaluate_bal_acc(dt_best, X_tr, y_tr, X_va, y_va, X_te, y_te)
rows.append({
    "Model": "Best DT (val-tuned)",
    "#Trees": 1,
    "max_depth": dt_best.get_params().get("max_depth", None),
    "BA_train": dt_b_tr, "BA_val": dt_b_va, "BA_test": dt_b_te
})

# Simple RF (from 3.1)
rows.append({
    "Model": "Simple RF (100, depth=3)",
    "#Trees": rf_simple.get_params().get("n_estimators", None),
    "max_depth": rf_simple.get_params().get("max_depth", None),
    "BA_train": rf_simple_tr, "BA_val": rf_simple_va, "BA_test": rf_simple_te
})

# Best RF (val-tuned) — as trained above on TRAIN only
rows.append({
    "Model": "Best RF (val-tuned)",
    "#Trees": rf_best_val.get_params().get("n_estimators", None),
    "max_depth": rf_best_val.get_params().get("max_depth", None),
    "BA_train": rf_bestval_tr, "BA_val": rf_bestval_va, "BA_test": rf_bestval_te
})

compare_df = pd.DataFrame(rows)
compare_df = compare_df[["Model", "#Trees", "max_depth", "BA_train", "BA_val", "BA_test"]]
compare_df.round(4)


,Model,#Trees,max_depth,BA_train,BA_val,BA_test
0,Simple DT,1,3,0.6458,0.6446,0.6458
1,Best DT (val-tuned),1,128,0.8324,0.7389,0.7145
2,"Simple RF (100, depth=3)",100,3,0.8210,0.7973,0.7829
3,Best RF (val-tuned),100,32,0.9646,0.8499,0.8408


In [54]:
# Retrain best RF on TRAIN+VAL, then evaluate on TEST
X_trva_full = _stackX(X_tr, X_va)
y_trva_full = np.hstack([y_tr, y_va])

rf_best_full = RandomForestClassifier(
    **best_params_rf, class_weight='balanced', n_jobs=-1
)
rf_best_full.fit(X_trva_full, y_trva_full)
ba_test_final = balanced_accuracy_score(y_te, rf_best_full.predict(X_te))
print(f"[Best RF retrained on train+val] Final TEST balanced accuracy: {ba_test_final:.4f}")


[Best RF retrained on train+val] Final TEST balanced accuracy: 0.8366


### 3.3 — Conclusions (Part C)

**Which model wins?**  
- The **Random Forest** clearly outperforms a single Decision Tree.  
- Even the **Simple RF (100 trees, depth=3)** typically **beats the Simple DT** on validation/test balanced accuracy.  
- The **Best RF (val-tuned)** achieves the **highest validation and test balanced accuracy** among all four models.  
  - *(Insert your numbers from the table: e.g., Best RF — BA_val = **…**, BA_test = **…**)*

**Why does RF win?**  
- RF averages many de-correlated trees (via bootstrapping + `max_features`), which **reduces variance** without increasing bias too much.  
- A single DT is **high-variance**; small changes in data can flip splits, hurting generalization.

**Overfitting vs. generalization**  
- The **train score** for DTs can be very high (especially for deeper trees), but the **gap to val/test** indicates overfitting.  
- RF narrows this gap due to ensembling, so **val/test performance is more stable**.  
- If your Best RF train ≫ val, consider increasing regularization (smaller `max_features`, shallower `max_depth`) or more data.

**Hyperparameter insights**  
- Your best RF used `max_features = **…**` and `max_depth = **…**` (from the grid).  
  - **Smaller `max_features`** promotes diversity (lower correlation between trees) → better averaging.  
  - **Moderate `max_depth`** balances bias and variance—too shallow underfits; too deep can overfit per-tree but is mitigated by the ensemble.  
- Increasing `n_estimators` mainly **reduces variance with diminishing returns**; beyond a point, accuracy plateaus while compute cost grows.

**Final model choice**  
- Retraining the **Best RF (val-tuned) on train+val** and reporting its test score is appropriate for the final submission.  
  - *(Insert your final score: Best RF retrained — BA_test = **…**)*

**Takeaway**  
- For this dataset, **ensemble methods (RF)** offer a **robust, higher-accuracy** solution than single trees, thanks to variance reduction via **bagging** and **feature subsampling**. The tuned RF is the recommended final model.
